# **Подготовка данных**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv2D, MaxPooling2D, BatchNormalization, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

# Скачивание и распаковка датасета
!wget https://storage.yandexcloud.net/aiueducation/Content/base/l4/bus.zip -q
!unzip -q bus.zip -d bus_dataset

# Путь к папкам
base_dir = 'bus_dataset'
categories = ['Входящий', 'Выходящий']
img_size = 64


# **Загрузка и обработка изображений**

In [ ]:
def load_data():
    data = []
    labels = []

    for category in categories:
        path = os.path.join(base_dir, category)
        class_num = categories.index(category)

        for img_name in os.listdir(path):
            try:
                img = Image.open(os.path.join(path, img_name)).convert('RGB').resize((img_size, img_size))
                data.append(np.array(img))
                labels.append(class_num)
            except Exception:
                pass

    return np.array(data) / 255.0, np.array(labels)

# Загружаем данные
X, y = load_data()

# Разделяем на обучающую и проверочную выборки
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f'Размер обучающей выборки:   {X_train.shape}')
print(f'Размер проверочной выборки: {X_val.shape}')


# **Создание модели**

Используем свёрточную нейронную сеть (CNN) — она работает с изображениями лучше, чем полносвязная, потому что учитывает пространственные связи между пикселями.


In [ ]:
model = Sequential([
    # Первый свёрточный блок
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(img_size, img_size, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Второй свёрточный блок
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Третий свёрточный блок
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Классификатор
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')   # 0 - входящий, 1 - выходящий
])

model.compile(optimizer=Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()


# **Обучение и результат**

In [ ]:
# EarlyStopping останавливает обучение, если точность перестала расти
early_stop = EarlyStopping(monitor='val_accuracy', mode='max', patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)

# Финальная точность
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print(f'\nТочность на проверочной выборке: {val_acc*100:.2f}%')
print(f'Порог 85% {"пройден ✅" if val_acc >= 0.85 else "НЕ пройден ❌"}')


# **Графики обучения**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'],     label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].axhline(0.85, color='red', linestyle='--', alpha=0.5, label='Порог 85%')
axes[0].set_title('Точность')
axes[0].set_xlabel('Эпоха')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'],     label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Ошибка')
axes[1].set_xlabel('Эпоха')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
